# ERA5 to PRISM Downscaling (Georgia)
Regional spatiotemporal downscaling with baseline progression from classical methods to ConvLSTM.


## 1. Problem Setup
ERA5 is coarse-resolution reanalysis data. PRISM is higher-resolution regional climate data.
The objective is to learn a mapping from ERA5 temporal windows to PRISM temperature targets.


## 2. Modeling Approach
Models used in this pipeline:
- persistence baseline
- linear baseline
- CNN (spatial)
- ConvLSTM (spatiotemporal)

The core question is whether adding temporal context improves downscaling quality.


## 3. Data Overview
ERA5 is loaded from NetCDF with dimensions like time/lat/lon. PRISM is loaded from daily raster grids.
Temporal windows are formed as `ERA5(t-k+1 ... t)` paired with `PRISM(t)` for matched dates.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import xarray as xr
import rioxarray
from IPython.display import Image, Markdown, display

ROOT = Path.cwd().resolve()
if not (ROOT / 'datasets').exists() and (ROOT.parent / 'datasets').exists():
    ROOT = ROOT.parent

ERA5_PATH = ROOT / 'data_raw/era5_georgia_temp.nc'
PRISM_DIR = ROOT / 'data_raw/prism'
RESULTS_DIR = ROOT / 'results/evaluation'

print(f'Project root: {ROOT}')
print(f'ERA5 path: {ERA5_PATH}')
print(f'PRISM path: {PRISM_DIR}')
print(f'Results path: {RESULTS_DIR}')
print(f'ERA5 available: {ERA5_PATH.exists()}')
print(f'PRISM available: {PRISM_DIR.exists()}')


In [ ]:
if ERA5_PATH.exists() and PRISM_DIR.exists():
    try:
        era5_ds = xr.open_dataset(ERA5_PATH)
        print('ERA5 variables:', list(era5_ds.data_vars))
        if era5_ds.data_vars:
            first_var = list(era5_ds.data_vars)[0]
            print(f'ERA5 {first_var} dims:', era5_ds[first_var].dims)
            print(f'ERA5 {first_var} shape:', tuple(era5_ds[first_var].shape))

        prism_files = sorted([p for p in PRISM_DIR.rglob('*') if p.suffix.lower() in {'.bil','.tif','.tiff','.asc'}])
        print('PRISM raster count:', len(prism_files))
        if prism_files:
            prism_da = rioxarray.open_rasterio(prism_files[0], masked=True)
            print('Sample PRISM raster:', prism_files[0].name)
            print('PRISM dims:', prism_da.dims)
            print('PRISM shape:', tuple(prism_da.shape))

        from datasets.prism_dataset import ERA5_PRISM_Dataset
        ds = ERA5_PRISM_Dataset(str(ERA5_PATH), str(PRISM_DIR), history_length=5, verbose=False)
        x, y = ds[0]
        print(f'Usable samples: {len(ds)}')
        print(f'Input shape [T, C, H, W]: {tuple(x.shape)}')
        print(f'Target shape [C, H, W]: {tuple(y.shape)}')
    except Exception as exc:
        print(f'Dataset load error: {exc}')
else:
    print('Data files are missing. Run:')
    print('python data_pipeline/download_era5_georgia.py --year 2023 --month 1')
    print('python data_pipeline/download_prism.py --start-date 20230101 --days 20 --variable tmean')


## 4. Model Comparison
Persistence is the low-effort reference. The linear baseline adds a global linear correction. CNN adds learned spatial mapping on each sample. ConvLSTM adds temporal state to use the ERA5 sequence directly.


## 5. Results
Loads saved outputs from `results/evaluation/`.


In [ ]:
cnn_images = sorted((RESULTS_DIR / 'cnn').glob('comparison_*.png'))
convlstm_images = sorted((RESULTS_DIR / 'convlstm').glob('comparison_*.png'))
cnn_metrics_path = RESULTS_DIR / 'cnn' / 'metrics.json'
convlstm_metrics_path = RESULTS_DIR / 'convlstm' / 'metrics.json'
summary_csv_path = RESULTS_DIR / 'baselines_summary.csv'
model_comparison_path = RESULTS_DIR / 'model_comparison.png'

if cnn_images:
    display(Markdown('### CNN output'))
    display(Image(filename=str(cnn_images[0])))
else:
    display(Markdown('CNN output not found.'))

if convlstm_images:
    display(Markdown('### ConvLSTM output'))
    display(Image(filename=str(convlstm_images[0])))
else:
    display(Markdown('ConvLSTM output not found.'))

if model_comparison_path.exists():
    display(Markdown('### Model comparison figure'))
    display(Image(filename=str(model_comparison_path)))

rows = []
for model_name, path in [('cnn', cnn_metrics_path), ('convlstm', convlstm_metrics_path)]:
    if path.exists():
        data = json.loads(path.read_text())
        rows.append({
            'model': model_name,
            'rmse': data.get('rmse'),
            'mae': data.get('mae'),
            'bias': data.get('bias'),
            'num_samples': data.get('num_samples'),
        })

if rows:
    metrics_df = pd.DataFrame(rows)
    display(Markdown('### Metrics from JSON'))
    display(metrics_df)
    if set(metrics_df['model']) == {'cnn', 'convlstm'}:
        cnn_rmse = float(metrics_df.loc[metrics_df['model']=='cnn', 'rmse'].iloc[0])
        conv_rmse = float(metrics_df.loc[metrics_df['model']=='convlstm', 'rmse'].iloc[0])
        cnn_mae = float(metrics_df.loc[metrics_df['model']=='cnn', 'mae'].iloc[0])
        conv_mae = float(metrics_df.loc[metrics_df['model']=='convlstm', 'mae'].iloc[0])
        display(Markdown(f'ConvLSTM vs CNN: ΔRMSE = {conv_rmse-cnn_rmse:+.4f}, ΔMAE = {conv_mae-cnn_mae:+.4f}'))

if summary_csv_path.exists():
    display(Markdown('### Baseline summary table'))
    display(pd.read_csv(summary_csv_path))

if not (cnn_images or convlstm_images or summary_csv_path.exists()):
    display(Markdown('Outputs are missing. Generate them with:'))
    print('python training/train_downscaler.py --model cnn --history-length 5 --epochs 20 --batch-size 4 --learning-rate 1e-3')
    print('python training/train_downscaler.py --model convlstm --history-length 5 --epochs 20 --batch-size 4 --learning-rate 1e-3')
    print('python evaluation/evaluate_model.py --models persistence linear cnn convlstm --history-length 5 --num-samples 8')


## 6. Metrics Interpretation
RMSE and MAE measure average prediction error in temperature units. Lower values indicate better fit to PRISM.
ConvLSTM should improve over CNN when enough aligned temporal samples are available. With limited data, gains can be small or unstable.


## 7. Visual Analysis
ERA5 fields are smoother than PRISM. PRISM contains finer spatial structure.
CNN usually captures large-scale structure but smooths local detail.
ConvLSTM can improve temporal consistency by using multiple ERA5 frames instead of a single snapshot.


## 8. Limitations
- limited number of aligned dates in typical local runs
- limited predictor variables
- no multi-source inputs yet
- compact model sizes


## 9. Future Work
- longer temporal windows
- more ERA5 variables
- multi-source inputs (remote sensing, drone imagery, terrain)
- uncertainty-aware models
